In [1]:
import pandas as pd
import numpy as np
from prophet import Prophet
from tqdm import tqdm
from functools import reduce
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import VotingRegressor

In [2]:
df=pd.read_csv("../ISI_dataset\merged_massor_reservoir.csv")
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:1: SyntaxWarning: invalid escape sequence '\m'
C:\Users\shrey\AppData\Local\Temp\ipykernel_26188\1804043079.py:1: SyntaxWarning: invalid escape sequence '\m'
  df=pd.read_csv("../ISI_dataset\merged_massor_reservoir.csv")


,state_name,crop_name,apy_item_interval_start,temperature_recorded_date,state_temperature_max_val,state_temperature_min_val,state_rainfall_val,yield,FRL,Live Cap FRL,Level,Current Live Storage
0,Chhattisgarh,masoor,2000,2000-01-01,28.95,8.16,0.0,0.27097,377.82,1.365667,349.965,1.2505
1,Chhattisgarh,masoor,2000,2000-01-02,28.95,7.38,0.0,0.27097,377.82,1.365667,349.920,1.2450
2,Chhattisgarh,masoor,2000,2000-01-03,28.15,5.39,0.0,0.27097,377.82,1.365667,349.885,1.2395
3,Chhattisgarh,masoor,2000,2000-01-04,28.13,6.17,0.0,0.27097,377.82,1.365667,349.840,1.2335
4,Chhattisgarh,masoor,2000,2000-01-05,28.02,4.52,0.0,0.27097,377.82,1.365667,349.800,1.2280


In [3]:
df['temperature_recorded_date'] = pd.to_datetime(df['temperature_recorded_date'])
df['year'] = df['temperature_recorded_date'].dt.year

In [4]:
# Use only data till 2022 for training
df = df[df['year'] < 2023].copy()

# Drop unreliable states
df = df[~df['state_name'].isin(['Odisha', 'Telangana'])]

# Group annually to match 2023 structure
df_annual = df.groupby(['state_name', 'crop_name', 'year']).agg({
    'state_rainfall_val': 'sum',
    'state_temperature_max_val': 'mean',
    'state_temperature_min_val': 'mean',
    'Live Cap FRL': 'mean',
    'FRL': 'mean',
    'Level': 'mean',
    'Current Live Storage': 'mean',
    'yield': 'mean'
}).reset_index()

In [5]:
df_annual.head()

,state_name,crop_name,year,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,yield
0,Chhattisgarh,masoor,2000,920.87,34.049372,17.556011,1.365667,377.82,344.059740,0.709251,0.27097
1,Chhattisgarh,masoor,2001,1558.68,33.983151,17.684027,1.365667,377.82,346.595055,1.009097,0.32372
2,Chhattisgarh,masoor,2002,1015.39,34.589452,17.793178,1.365667,377.82,347.156137,0.871140,0.30895
3,Chhattisgarh,masoor,2003,1643.77,34.080603,18.244548,1.365667,377.82,346.964562,1.039784,0.34193
4,Chhattisgarh,masoor,2004,1154.23,34.024344,17.587814,1.365667,377.82,351.123019,1.426123,0.25837


In [6]:
# One-hot encode 'state_name'
df_encoded = pd.get_dummies(df_annual, columns=['state_name'])
df_encoded['year']=df_encoded['year']-2000
# Define features: original + one-hot encoded state columns
state_columns = [col for col in df_encoded.columns if col.startswith('state_name_')]

In [7]:
# Define features and target
features = ['state_rainfall_val', 'state_temperature_max_val', 'state_temperature_min_val', 'Live Cap FRL', 'FRL','Level','Current Live Storage']+ state_columns

# Split manually by year
train_df = df_encoded[df_encoded['year'] <= 20]
test_df = df_encoded[df_encoded['year'].between(21, 22)]

In [8]:
X_train = train_df[features]
y_train = train_df['yield']
X_test = test_df[features]
y_test = test_df['yield']

In [9]:
# Models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR()
}

# Results container
results = []

# Loop through models
for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Metrics
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    results.append({
        'Model': name,
        'Train R²': round(train_r2, 4),
        'Test R²': round(test_r2, 4),
        'Train RMSE': round(train_rmse, 2),
        'Test RMSE': round(test_rmse, 2)
    })

# Display results
results_df = pd.DataFrame(results)
print(results_df.sort_values(by='Test R²', ascending=False))

                      Model  Train R²  Test R²  Train RMSE  Test RMSE
3         Gradient Boosting    0.9557  -0.7095        0.06       0.47
0         Linear Regression    0.5924  -0.8951        0.17       0.50
2                   XGBoost    1.0000  -0.9617        0.00       0.51
1             Random Forest    0.9269  -0.9731        0.07       0.51
4  Support Vector Regressor    0.2640  -1.0229        0.23       0.51


In [10]:
# Initialize individual models
lr = LinearRegression()
svr = SVR()
xgb = XGBRegressor(random_state=42)

# Ensemble model
ensemble = VotingRegressor(estimators=[
    ('lr', lr),
    ('svr', svr),
    ('xgb', xgb)
])

# Fit ensemble
ensemble.fit(X_train, y_train)

# Predict
train_pred = ensemble.predict(X_train)
test_pred = ensemble.predict(X_test)

# Evaluate
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("📊 Ensemble Performance:")
print(f"Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")


📊 Ensemble Performance:
Train R²: 0.7793, Test R²: -0.9044
Train RMSE: 0.1237, Test RMSE: 0.4979


In [11]:
# --- 1. Define function to forecast any single feature using Prophet ---
def forecast_feature_prophet(df, feature_name):
    forecast_data = []

    for (state, crop), group in tqdm(df.groupby(['state_name', 'crop_name'])):
        yearly_data = group.groupby('year')[feature_name].mean().reset_index()

        if yearly_data.shape[0] < 4:
            continue

        prophet_df = yearly_data.rename(columns={'year': 'ds', feature_name: 'y'})
        prophet_df['ds'] = pd.to_datetime(prophet_df['ds'], format='%Y')

        try:
            model = Prophet()
            model.fit(prophet_df)

            future = pd.DataFrame({'ds': [pd.to_datetime('2023')]})
            forecast = model.predict(future)
            yhat = forecast['yhat'].values[0]

            forecast_data.append({
                'state_name': state,
                'crop_name': crop,
                feature_name: yhat
            })
        except:
            continue

    return pd.DataFrame(forecast_data)

# --- 2. Forecast each feature separately ---
df_rain = forecast_feature_prophet(df, 'state_rainfall_val')
df_temp_max = forecast_feature_prophet(df, 'state_temperature_max_val')
df_temp_min = forecast_feature_prophet(df, 'state_temperature_min_val')
df_livecap = forecast_feature_prophet(df, 'Live Cap FRL')
df_frl = forecast_feature_prophet(df, 'FRL')
df_level = forecast_feature_prophet(df, 'Level')
df_cls = forecast_feature_prophet(df, 'Current Live Storage')

# --- 3. Merge all forecasted dataframes ---
from functools import reduce
dfs = [df_rain, df_temp_max, df_temp_min, df_livecap, df_frl, df_level, df_cls]
df_2023 = reduce(lambda left, right: pd.merge(left, right, on=['state_name', 'crop_name'], how='outer'), dfs)

# --- 4. One-hot encode state_name ---
df_2023_encoded = df_2023.copy()  # Keep original columns
state_names = df_2023_encoded[['state_name', 'crop_name']]  # Keep for merging later

df_2023_encoded = pd.get_dummies(df_2023_encoded, columns=['state_name'])
df_2023_encoded = pd.concat([state_names, df_2023_encoded.drop(columns=['crop_name'])], axis=1)


  0%|          | 0/7 [00:00<?, ?it/s]19:08:47 - cmdstanpy - INFO - Chain [1] start processing
19:08:47 - cmdstanpy - INFO - Chain [1] done processing
 14%|█▍        | 1/7 [00:01<00:11,  1.86s/it]19:08:48 - cmdstanpy - INFO - Chain [1] start processing
19:08:48 - cmdstanpy - INFO - Chain [1] done processing
 29%|██▊       | 2/7 [00:02<00:04,  1.05it/s]19:08:48 - cmdstanpy - INFO - Chain [1] start processing
19:08:48 - cmdstanpy - INFO - Chain [1] done processing
 43%|████▎     | 3/7 [00:02<00:02,  1.51it/s]19:08:48 - cmdstanpy - INFO - Chain [1] start processing
19:08:48 - cmdstanpy - INFO - Chain [1] done processing
 57%|█████▋    | 4/7 [00:02<00:01,  1.87it/s]19:08:49 - cmdstanpy - INFO - Chain [1] start processing
19:08:49 - cmdstanpy - INFO - Chain [1] done processing
 71%|███████▏  | 5/7 [00:03<00:00,  2.23it/s]19:08:49 - cmdstanpy - INFO - Chain [1] start processing
19:08:49 - cmdstanpy - INFO - Chain [1] done processing
 86%|████████▌ | 6/7 [00:03<00:00,  2.58it/s]19:08:49 - cmds

In [12]:
df_2023_encoded.head()

,state_name,crop_name,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,state_name_Chhattisgarh,state_name_Jharkhand,state_name_Madhya Pradesh,state_name_Rajasthan,state_name_Uttar Pradesh,state_name_Uttarakhand,state_name_West Bengal
0,Chhattisgarh,masoor,3.704114,33.805702,18.200942,1.365667,377.820000,353.377518,1.355013,True,False,False,False,False,False,False
1,Jharkhand,masoor,3.401457,33.196613,18.388643,0.404750,296.267500,290.842515,0.200967,False,True,False,False,False,False,False
2,Madhya Pradesh,masoor,3.151770,35.311940,17.200250,2.744818,373.365455,327.569102,1.675670,False,False,True,False,False,False,False
3,Rajasthan,masoor,2.096794,35.089229,17.012664,1.165400,287.252000,271.164415,0.750774,False,False,False,True,False,False,False
4,Uttar Pradesh,masoor,2.244383,34.047351,16.984807,2.898000,183.210471,160.237735,0.722527,False,False,False,False,True,False,False


In [13]:
X_2023 = df_2023_encoded[features]

# Use your trained ensemble model
y_2023_pred = ensemble.predict(X_2023)

# Add prediction to the dataframe
df_2023_encoded['predicted_yield'] = y_2023_pred

# Select output
output_2023 = df_2023_encoded[['crop_name'] + [col for col in df_2023_encoded.columns if col.startswith('state_name_')] + ['predicted_yield']]


In [14]:
# Convert dummy columns back to state_name
state_names = df_2023_encoded[[col for col in df_2023_encoded.columns if col.startswith('state_name_')]].idxmax(axis=1)
state_names = state_names.str.replace('state_name_', '')

# Final output
final_2023_yield = pd.DataFrame({
    'state_name': state_names,
    'crop_name': df_2023_encoded['crop_name'],
    'predicted_yield_2023': df_2023_encoded['predicted_yield']
})

print(final_2023_yield)
# final_2023_yield.to_csv("../yield_prediction.csv", index=False)

       state_name crop_name  predicted_yield_2023
0    Chhattisgarh    masoor              0.611277
1       Jharkhand    masoor              0.920380
2  Madhya Pradesh    masoor              0.888307
3       Rajasthan    masoor              0.982795
4   Uttar Pradesh    masoor              0.885087
5     Uttarakhand    masoor              0.844196
6     West Bengal    masoor              0.881192


In [15]:
# Step 1: Get actual yields from 2019 to 2022
df_recent = df_annual[df_annual['year'].between(2019, 2022)].copy()

# Pivot to get each year's yield as a column
yield_table = df_recent.pivot_table(
    index=['state_name', 'crop_name'],
    columns='year',
    values='yield'
).reset_index()

# Rename columns for clarity
yield_table = yield_table.rename(columns={
    2019: 'yield_2019',
    2020: 'yield_2020',
    2021: 'yield_2021',
    2022: 'yield_2022'
})

# Step 2: Prepare 2023 predicted yield
df_2023_yield = df_2023_encoded[['state_name', 'crop_name', 'predicted_yield']].copy()
df_2023_yield = df_2023_yield.rename(columns={'predicted_yield': 'yield_2023'})

# Step 3: Merge the 2023 predicted yield into the table
final_yield_table = pd.merge(yield_table, df_2023_yield, on=['state_name', 'crop_name'], how='left')

# Display final table
print(final_yield_table)


       state_name crop_name  yield_2019  yield_2020  yield_2021  yield_2022  \
0    Chhattisgarh    masoor     0.32564     0.40361     0.38849     2.01897   
1       Jharkhand    masoor     0.84918     0.88018     0.88038     0.81539   
2  Madhya Pradesh    masoor     0.77657     1.13529     1.17985     1.14960   
3       Rajasthan    masoor     1.37547     1.34721     1.32602     1.37483   
4   Uttar Pradesh    masoor     0.97819     1.01380     0.94447     0.94985   
5     Uttarakhand    masoor     1.01552     0.75687     0.87254     0.92518   
6     West Bengal    masoor     0.83588     0.89715     0.76937     0.95965   

   yield_2023  
0    0.611277  
1    0.920380  
2    0.888307  
3    0.982795  
4    0.885087  
5    0.844196  
6    0.881192  


<!-- leakage-safe-evaluation -->
## Leakage-safe evaluation

This section evaluates the **massor** yield model as a rolling one-year-ahead
forecast. Models are selected using five-year walk-forward validation on data
through 2020 and evaluated once on the untouched 2021-2022 holdout.

- **Previous-year baseline:** predicts the current yield from the previous
  calendar year's observed yield.
- **Lagged features:** previous-year yield and the trailing three-year mean;
  both use only earlier targets.
- **WAPE:** total absolute error divided by total actual yield. Lower is better.
- **Normalized RMSE:** RMSE divided by mean absolute yield. Lower is better.

Observed weather and reservoir variables are used for the evaluation year, so
these results evaluate the yield-regression stage rather than the complete
Prophet-to-yield forecasting pipeline.

In [1]:
from pathlib import Path
import os
import sys
from IPython.display import display
import pandas as pd

repo_root = Path.cwd().resolve()
while not (repo_root / "evaluate_models.py").is_file():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not locate evaluate_models.py")
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from evaluate_models import (  # noqa: E402
    DATASETS,
    build_models,
    evaluate_dataset,
    prepare_annual_data,
)

crop_key = "massor"
spec = DATASETS[crop_key]
data_dir = Path(os.environ.get("ISI_DATA_DIR", repo_root / "ISI_dataset"))
annual_evaluation_data = prepare_annual_data(
    data_dir / spec.filename, spec.excluded_states
)

In [2]:
evaluation_summary, fold_results, cv_results, holdout_predictions = evaluate_dataset(
    crop_key,
    annual_evaluation_data,
    build_models(),
    holdout_start=2021,
    holdout_end=2022,
    cv_years=5,
    min_train_years=5,
)

print("Walk-forward validation (sorted by RMSE):")
display(
    cv_results[["model", "folds", "r2", "mae", "rmse", "nrmse_pct", "wape_pct"]]
    .round(4)
)

print("2021-2022 holdout result:")
display(
    pd.DataFrame([evaluation_summary])[
        [
            "selected_model",
            "holdout_samples",
            "holdout_r2",
            "holdout_mae",
            "holdout_rmse",
            "holdout_nrmse_pct",
            "holdout_wape_pct",
            "baseline_rmse",
            "rmse_improvement_vs_baseline_pct",
        ]
    ].round(4)
)

Walk-forward validation (sorted by RMSE):


,model,folds,r2,mae,rmse,nrmse_pct,wape_pct
0,Random Forest,5,0.5574,0.1394,0.1740,19.8594,15.9172
1,Gradient Boosting,5,0.5564,0.1427,0.1741,19.8808,16.2914
2,SVR,5,0.5262,0.1398,0.1800,20.5467,15.9555
3,Voting Ensemble,5,0.5249,0.1426,0.1802,20.5745,16.2759
4,Linear Regression,5,0.5139,0.1380,0.1823,20.8124,15.7575
5,XGBoost,5,0.3459,0.1641,0.2115,24.1415,18.7356


2021-2022 holdout result:


,selected_model,holdout_samples,holdout_r2,holdout_mae,holdout_rmse,holdout_nrmse_pct,holdout_wape_pct,baseline_rmse,rmse_improvement_vs_baseline_pct
0,Random Forest,14.0,-0.625,0.2264,0.4599,44.2406,21.781,0.4426,-3.9226
